Loading tables for Exploratory Data Analysis

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

sales = spark.table("retail_inv.bronze.sales_wide")
calendar = spark.table("retail_inv.bronze.calendar")
prices = spark.table("retail_inv.bronze.sell_prices")

# Identify daily sales columns and sort them numerically.
day_cols = sorted(
    [
        c for c in sales.columns
        if c.startswith("d_") and c[2:].isdigit()
    ],
    key=lambda c: int(c[2:])
)

print("Sales rows:", sales.count())
print("Daily sales columns:", len(day_cols))
print("Price rows:", prices.count())
print("Calendar rows:", calendar.count())

Sales rows: 30490
Daily sales columns: 1913
Price rows: 6841121
Calendar rows: 1969


What share of item-store-days have zero sales?

In [0]:
# Put daily values into an array for each item-store row.
daily = sales.select(
    F.array(*[F.col(c) for c in day_cols]).alias("daily_sales")
)

# Count zero-sales days and observed (non-null) days for each row.
row_counts = daily.select(
    F.aggregate(
        "daily_sales",
        F.lit(0).cast("long"),
        lambda acc, x: acc + F.when(
            x == 0, F.lit(1).cast("long")
        ).otherwise(F.lit(0).cast("long"))
    ).alias("zero_days"),

    F.aggregate(
        "daily_sales",
        F.lit(0).cast("long"),
        lambda acc, x: acc + F.when(
            x.isNotNull(), F.lit(1).cast("long")
        ).otherwise(F.lit(0).cast("long"))
    ).alias("observed_days")
)

result = row_counts.agg(
    F.sum("zero_days").alias("zero_item_store_days"),
    F.sum("observed_days").alias("observed_item_store_days")
).first()

zero_days = result["zero_item_store_days"]
observed_days = result["observed_item_store_days"]

zero_pct = (
    zero_days / observed_days * 100
    if observed_days else 0
)

print(f"Zero-sales item-store-days: {zero_days:,}")
print(f"Observed item-store-days: {observed_days:,}")
print(f"Zero-sales share: {zero_pct:.2f}%")

Zero-sales item-store-days: 39,777,094
Observed item-store-days: 58,327,370
Zero-sales share: 68.20%


How many item-stores have no price in the early weeks?

In [0]:
# All product-store combinations present in the sales dataset.
item_stores = sales.select(
    "item_id", "store_id"
).distinct()

# First four calendar weeks in the dataset.
early_weeks = (
    calendar.select("wm_yr_wk")
    .distinct()
    .orderBy("wm_yr_wk")
    .limit(4)
)

early_week_values = [
    r["wm_yr_wk"] for r in early_weeks.collect()
]

# Product-store combinations with a price in at least one early week.
early_price_pairs = (
    prices.join(F.broadcast(early_weeks), on="wm_yr_wk", how="inner")
    .select("item_id", "store_id")
    .distinct()
)

# Product-store combinations with any recorded price in the full dataset.
all_price_pairs = prices.select(
    "item_id", "store_id"
).distinct()

# Combinations with no price record during any of the first four weeks.
missing_early = item_stores.join(
    early_price_pairs,
    on=["item_id", "store_id"],
    how="left_anti"
)

total_pairs = item_stores.count()
missing_early_count = missing_early.count()

# Separate later-priced pairs from those never priced in this file.
priced_at_some_point = missing_early.join(
    all_price_pairs,
    on=["item_id", "store_id"],
    how="inner"
).count()

never_priced = missing_early.join(
    all_price_pairs,
    on=["item_id", "store_id"],
    how="left_anti"
).count()

print("First four weeks:", early_week_values)
print(f"Total item-store pairs: {total_pairs:,}")
print(f"No price in early weeks: {missing_early_count:,}")
print(
    f"Share with no early price: "
    f"{missing_early_count / total_pairs * 100:.2f}%"
    if total_pairs else "No item-store pairs found."
)
print(f"No early price, but priced later: {priced_at_some_point:,}")
print(f"No price anywhere in the price file: {never_priced:,}")

First four weeks: [11101, 11102, 11103, 11104]
Total item-store pairs: 30,490
No price in early weeks: 17,691
Share with no early price: 58.02%
No early price, but priced later: 17,691
No price anywhere in the price file: 0


How often do prices change per item?

In [0]:
price_window = Window.partitionBy(
    "store_id", "item_id"
).orderBy("wm_yr_wk")

price_history = (
    prices
    .withColumn(
        "previous_price",
        F.lag("sell_price").over(price_window)
    )
    .withColumn(
        "price_changed",
        F.when(
            F.col("previous_price").isNotNull()
            & (F.col("sell_price") != F.col("previous_price")),
            1
        ).otherwise(0)
    )
)

# Count price changes for each item-store pair.
pair_changes = price_history.groupBy(
    "store_id", "item_id"
).agg(
    F.sum("price_changed").alias("price_change_count"),
    F.count("*").alias("weeks_with_price")
)

overall_changes = pair_changes.agg(
    F.count("*").alias("item_store_pairs"),
    F.sum("price_change_count").alias("total_price_changes"),
    F.avg("price_change_count").alias("avg_changes_per_pair"),
    F.expr(
        "percentile_approx(price_change_count, 0.5)"
    ).alias("median_changes_per_pair"),
    F.avg(
        F.when(F.col("price_change_count") > 0, 1.0).otherwise(0.0)
    ).alias("share_of_pairs_with_changes")
).first()

print(f"Item-store pairs: {overall_changes['item_store_pairs']:,}")
print(f"Total recorded price changes: {overall_changes['total_price_changes']:,}")
print(
    f"Average changes per item-store pair: "
    f"{overall_changes['avg_changes_per_pair']:.2f}"
)
print(
    f"Median changes per item-store pair: "
    f"{overall_changes['median_changes_per_pair']}"
)
print(
    f"Pairs with at least one price change: "
    f"{overall_changes['share_of_pairs_with_changes'] * 100:.2f}%"
)

# Summarize price changes by item across its stores.
item_changes = pair_changes.groupBy("item_id").agg(
    F.sum("price_change_count").alias("changes_across_stores"),
    F.avg("price_change_count").alias("avg_changes_per_store"),
    F.countDistinct("store_id").alias("number_of_stores")
)

display(
    item_changes
    .orderBy(F.desc("changes_across_stores"))
    .limit(10)
)

Item-store pairs: 30,490
Total recorded price changes: 78,815
Average changes per item-store pair: 2.58
Median changes per item-store pair: 2
Pairs with at least one price change: 72.95%


item_id,changes_across_stores,avg_changes_per_store,number_of_stores
FOODS_3_822,198,19.8,10
FOODS_3_615,180,18.0,10
FOODS_3_274,179,17.9,10
FOODS_3_029,169,16.9,10
FOODS_1_095,166,16.6,10
FOODS_1_129,166,16.6,10
FOODS_3_432,165,16.5,10
FOODS_3_090,164,16.4,10
FOODS_1_096,164,16.4,10
FOODS_3_738,159,15.9,10


Which categories sell the most?

In [0]:
# Calculate total units sold over all observed days for each item-store row.
row_sales = sales.select(
    "cat_id",
    F.aggregate(
        F.array(*[F.col(c) for c in day_cols]),
        F.lit(0).cast("long"),
        lambda acc, x: acc + F.coalesce(
            x.cast("long"),
            F.lit(0).cast("long")
        )
    ).alias("total_units_sold")
)

category_sales = row_sales.groupBy("cat_id").agg(
    F.sum("total_units_sold").alias("units_sold")
)

all_units = category_sales.agg(
    F.sum("units_sold").alias("all_units")
).first()["all_units"]

category_sales = (
    category_sales
    .withColumn(
        "share_pct",
        F.round(F.col("units_sold") / F.lit(all_units) * 100, 2)
    )
    .orderBy(F.desc("units_sold"))
)

display(category_sales)

cat_id,units_sold,share_pct
FOODS,45089939,68.63
HOUSEHOLD,14480670,22.04
HOBBIES,6124800,9.32


Validate prices and split zero-sales days

In [0]:
sales = spark.table("retail_inv.bronze.sales_wide")
calendar = spark.table("retail_inv.bronze.calendar")
prices = spark.table("retail_inv.bronze.sell_prices")

keys = ["item_id", "store_id", "wm_yr_wk"]

# First ensure the weekly price join won't duplicate sales rows.
duplicate_price_keys = (
    prices.groupBy(*keys)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate price keys:", duplicate_price_keys)

if duplicate_price_keys > 0:
    print("STOP: inspect duplicate price keys before continuing.")
else:
    # Convert the daily sales columns into item-store-day rows.
    day_cols = sorted(
        [
            c for c in sales.columns
            if c.startswith("d_") and c[2:].isdigit()
        ],
        key=lambda c: int(c[2:])
    )

    stack_args = ", ".join(
        part
        for c in day_cols
        for part in (f"'{c}'", f"`{c}`")
    )

    long_sales = sales.selectExpr(
        "item_id",
        "store_id",
        f"stack({len(day_cols)}, {stack_args}) AS (d, units)"
    )

    daily = long_sales.join(
        calendar.select("d", "wm_yr_wk"),
        on="d",
        how="inner"
    )

    # First week with a recorded price for each item-store pair.
    first_price = prices.groupBy(
        "item_id", "store_id"
    ).agg(
        F.min("wm_yr_wk").alias("first_price_week")
    )

    # Add the first price week and the price recorded for each week.
    daily_priced = (
        daily
        .join(first_price, ["item_id", "store_id"], "left")
        .join(
            prices.select(*keys, "sell_price"),
            on=keys,
            how="left"
        )
    )

    before_first_price = (
        F.col("wm_yr_wk") < F.col("first_price_week")
    )
    has_price = F.col("sell_price").isNotNull()
    zero_sales = F.col("units") == 0
    positive_sales = F.col("units") > 0

    metrics = daily_priced.agg(
        F.count("*").alias("joined_item_store_days"),

        F.sum(
            F.when(
                F.col("sell_price").isNull(), 1
            ).otherwise(0)
        ).alias("days_without_price"),

        F.sum(
            F.when(
                F.col("sell_price").isNull() & positive_sales, 1
            ).otherwise(0)
        ).alias("positive_sales_days_without_price"),

        F.sum(
            F.when(
                F.col("sell_price").isNull() & positive_sales,
                F.col("units")
            ).otherwise(0)
        ).alias("units_sold_without_price"),

        F.sum(
            F.when(before_first_price, 1).otherwise(0)
        ).alias("days_before_first_price"),

        F.sum(
            F.when(
                before_first_price & zero_sales, 1
            ).otherwise(0)
        ).alias("zero_days_before_first_price"),

        F.sum(
            F.when(has_price, 1).otherwise(0)
        ).alias("days_with_recorded_price"),

        F.sum(
            F.when(has_price & zero_sales, 1).otherwise(0)
        ).alias("zero_days_with_recorded_price")
    ).first()

    print("\n--- Price coverage validation ---")
    print("Item-store-days:", f"{metrics['joined_item_store_days']:,}")
    print("Days without price:", f"{metrics['days_without_price']:,}")
    print(
        "Positive-sales days without price:",
        f"{metrics['positive_sales_days_without_price']:,}"
    )
    print(
        "Units sold without price:",
        f"{metrics['units_sold_without_price']:,}"
    )

    print("\n--- Zero-sales split ---")
    print(
        "Days before first recorded price:",
        f"{metrics['days_before_first_price']:,}"
    )
    print(
        "Zero-sales share before first price:",
        (
            f"{metrics['zero_days_before_first_price'] / metrics['days_before_first_price'] * 100:.2f}%"
            if metrics["days_before_first_price"] else "N/A"
        )
    )
    print(
        "Days with recorded prices:",
        f"{metrics['days_with_recorded_price']:,}"
    )
    print(
        "Zero-sales share on days with recorded prices:",
        (
            f"{metrics['zero_days_with_recorded_price'] / metrics['days_with_recorded_price'] * 100:.2f}%"
            if metrics["days_with_recorded_price"] else "N/A"
        )
    )

Duplicate price keys: 0

--- Price coverage validation ---
Item-store-days: 58,327,370
Days without price: 12,299,413
Positive-sales days without price: 0
Units sold without price: 0

--- Zero-sales split ---
Days before first recorded price: 12,299,413
Zero-sales share before first price: 100.00%
Days with recorded prices: 46,027,957
Zero-sales share on days with recorded prices: 59.70%


calculate revenue by category

In [0]:
sales = spark.table("retail_inv.bronze.sales_wide")
calendar = spark.table("retail_inv.bronze.calendar")
prices = spark.table("retail_inv.bronze.sell_prices")

day_cols = sorted(
    [c for c in sales.columns if c.startswith("d_") and c[2:].isdigit()],
    key=lambda c: int(c[2:])
)

stack_args = ", ".join(
    part
    for c in day_cols
    for part in (f"'{c}'", f"`{c}`")
)

# Expand sales into daily records.
daily_sales = sales.selectExpr(
    "item_id",
    "store_id",
    "cat_id",
    f"stack({len(day_cols)}, {stack_args}) AS (d, units)"
)

# Map each sales day to its calendar week.
daily_sales = daily_sales.join(
    calendar.select("d", "wm_yr_wk"),
    on="d",
    how="inner"
)

# Aggregate units by item, store, category, and week.
weekly_sales = daily_sales.groupBy(
    "item_id", "store_id", "cat_id", "wm_yr_wk"
).agg(
    F.sum("units").alias("weekly_units")
)

# Join weekly unit sales to the recorded weekly price.
weekly_revenue = weekly_sales.join(
    prices.select("item_id", "store_id", "wm_yr_wk", "sell_price"),
    on=["item_id", "store_id", "wm_yr_wk"],
    how="left"
)

# Check that sold units are not missing a price.
unpriced_sales = weekly_revenue.filter(
    (F.col("weekly_units") > 0) & F.col("sell_price").isNull()
).count()

print("Weekly sales rows missing a price despite positive sales:", unpriced_sales)

if unpriced_sales == 0:
    category_revenue = weekly_revenue.groupBy("cat_id").agg(
        F.sum(
            F.col("weekly_units") * F.coalesce(F.col("sell_price"), F.lit(0.0))
        ).alias("estimated_revenue")
    )

    revenue_total = category_revenue.agg(
        F.sum("estimated_revenue").alias("total")
    ).first()["total"]

    category_revenue = (
        category_revenue
        .withColumn(
            "revenue_share_pct",
            F.round(F.col("estimated_revenue") / F.lit(revenue_total) * 100, 2)
        )
        .orderBy(F.desc("estimated_revenue"))
    )

    display(category_revenue)
else:
    print("Investigate missing prices before calculating revenue.")

Weekly sales rows missing a price despite positive sales: 0


cat_id,estimated_revenue,revenue_share_pct
FOODS,1.0890462204004636E8,58.03
HOUSEHOLD,5.595354268000687E7,29.81
HOBBIES,2.2818405299993165E7,12.16


weekday, monthly and store-level patterns

In [0]:
sales = spark.table("retail_inv.bronze.sales_wide")
calendar = spark.table("retail_inv.bronze.calendar")

day_cols = sorted(
    [c for c in sales.columns if c.startswith("d_") and c[2:].isdigit()],
    key=lambda c: int(c[2:])
)

stack_args = ", ".join(
    part
    for c in day_cols
    for part in (f"'{c}'", f"`{c}`")
)

# Convert the wide sales data into daily records.
daily_sales = sales.selectExpr(
    "item_id",
    "store_id",
    f"stack({len(day_cols)}, {stack_args}) AS (d, units)"
)

# Attach calendar attributes to every sales day.
daily_sales = daily_sales.join(
    calendar.select("d", "date", "weekday", "wday", "month", "year"),
    on="d",
    how="inner"
)

# 1. Total units sold by store.
store_sales = (
    daily_sales
    .groupBy("store_id")
    .agg(F.sum("units").alias("units_sold"))
    .orderBy(F.desc("units_sold"))
)

print("Sales by store")
display(store_sales)

# Aggregate all items and stores to one total per calendar day.
daily_totals = daily_sales.groupBy(
    "d", "date", "weekday", "wday", "month", "year"
).agg(
    F.sum("units").alias("total_units")
)

# 2. Average daily units sold by weekday.
weekday_sales = daily_totals.groupBy(
    "weekday", "wday"
).agg(
    F.avg("total_units").alias("avg_units_per_day"),
    F.sum("total_units").alias("total_units")
).orderBy("wday")

print("Sales by weekday")
display(weekday_sales)

# 3. Total units sold by calendar month.
monthly_sales = daily_totals.groupBy(
    "year", "month"
).agg(
    F.sum("total_units").alias("total_units")
).orderBy("year", "month")

print("Monthly sales trend")
display(monthly_sales)

Sales by store


store_id,units_sold
CA_3,11188180
CA_1,7698216
TX_2,7214384
WI_2,6544012
WI_3,6427782
TX_3,6089330
CA_2,5685475
TX_1,5595292
WI_1,5149062
CA_4,4103676


Sales by weekday


weekday,wday,avg_units_per_day,total_units
Saturday,1,41546.89416058394,11383849
Sunday,2,41130.02189781022,11269626
Monday,3,32852.96703296703,8968860
Tuesday,4,30368.78021978022,8290677
Wednesday,5,30010.021978021978,8192736
Thursday,6,30205.007326007326,8245967
Friday,7,34225.98534798535,9343694


Monthly sales trend


year,month,total_units
2011,1,88163
2011,2,726375
2011,3,763567
2011,4,737713
2011,5,719562
2011,6,753380
2011,7,819820
2011,8,825694
2011,9,813683
2011,10,898243


In [0]:
# Find the first and last dates represented in daily sales.
bounds = daily_totals.agg(
    F.min("date").alias("first_date"),
    F.max("date").alias("last_date")
).first()

first_date = bounds["first_date"]
last_date = bounds["last_date"]

# Exclude partial months at the beginning and end of the dataset.
full_months = monthly_sales.filter(
    ~(
        ((F.col("year") == first_date.year)
         & (F.col("month") == first_date.month))
        |
        ((F.col("year") == last_date.year)
         & (F.col("month") == last_date.month))
    )
)

print("Highest-sales complete months:")
display(
    full_months
    .orderBy(F.desc("total_units"))
    .limit(5)
)

print("Lowest-sales complete months:")
display(
    full_months
    .orderBy("total_units")
    .limit(5)
)

Highest-sales complete months:


year,month,total_units
2016,3,1275746
2015,8,1254341
2015,10,1248437
2015,7,1232147
2016,1,1215286


Lowest-sales complete months:


year,month,total_units
2011,5,719562
2011,2,726375
2011,4,737713
2011,6,753380
2011,3,763567
